# Qwen-Image-2.1 on Colab (L4)

ComfyUI をバックグラウンドで動かし、**このノートブックのセルから**画像を生成します。

**使い方**：「ランタイム → ランタイムのタイプを変更」で **L4 GPU** を選び、上から順にセルを実行 → 「4. 画像を生成」のフォームを書き換えて何度でも実行。

| 部品 | ファイル | サイズ |
|---|---|---|
| DiT | `qwen_image_2.1-Q8_0.gguf`（pottokao/Qwen-Image-2.1-DiT-GGUF） | 7.7 GB |
| テキストエンコーダー | Heretic FP8（既定）または 公式 BF16 | 9.3 / 17.5 GB |
| VAE | `qwen_image_2.1_vae_bf16.safetensors`（Comfy-Org/Qwen-Image-2.1） | 0.7 GB |

> **注意**
> - Qwen-Image-2.1 本体（DiT / VAE）は **qwen-research ライセンス**です。個人の試用・研究向けで、商用や業務利用は条件を確認してください。
> - Heretic は拒否応答を除去した派生エンコーダーです。業務検証では「公式 BF16」を選んでください。


In [ ]:
#@title 1. GPU とメモリを確認
!nvidia-smi --query-gpu=name,memory.total --format=csv
import psutil
print(f"System RAM: {psutil.virtual_memory().total / 1e9:.1f} GB")


In [ ]:
#@title 2. ComfyUI とカスタムノードをインストール（数分）
import os, subprocess

ROOT = "/content/ComfyUI"

def sh(cmd, cwd=ROOT):
    subprocess.run(cmd, shell=True, cwd=cwd, check=True)

os.makedirs(ROOT, exist_ok=True)
if not os.path.isdir(f"{ROOT}/.git"):
    # フォルダが先にできていても（モデルを先に落とした等）、その上に ComfyUI を展開する
    sh("git init -q && git remote add origin https://github.com/comfyanonymous/ComfyUI")
    sh("git fetch -q --depth 1 origin master && git checkout -q -f FETCH_HEAD")

for repo in ["city96/ComfyUI-GGUF", "pottokao-dotcom/ComfyUI-GGUF-Qwen3VL-TE"]:
    d = f"{ROOT}/custom_nodes/{repo.split('/')[1]}"
    if not os.path.isdir(f"{d}/.git"):
        sh(f"rm -rf {d} && git clone -q --depth 1 https://github.com/{repo} {d}")

sh("pip install -q -r requirements.txt -r custom_nodes/ComfyUI-GGUF/requirements.txt")
sh("pip install -q -U huggingface_hub hf_xet")
sh('git log -1 --format="ComfyUI commit: %h (%cd)"')
print("インストール完了")


In [ ]:
#@title 3. モデルをダウンロード（初回 5〜10 分）
TEXT_ENCODER = "Heretic FP8" #@param ["Heretic FP8", "公式 BF16"]

import os
from huggingface_hub import hf_hub_download

M = "/content/ComfyUI/models"
DIT = "qwen_image_2.1-Q8_0.gguf"
VAE = "qwen_image_2.1_vae_bf16.safetensors"

hf_hub_download("pottokao/Qwen-Image-2.1-DiT-GGUF", DIT, local_dir=f"{M}/diffusion_models")
hf_hub_download("Comfy-Org/Qwen-Image-2.1", f"vae/{VAE}", local_dir=M)

if TEXT_ENCODER == "Heretic FP8":
    TE = "qwen3vl_8b_fp8_heretic.safetensors"
    hf_hub_download("pottokao/Qwen-Image-2.1-Text-Encoder-Heretic-GGUF", TE, local_dir=f"{M}/text_encoders")
else:
    TE = "qwen3vl_8b_bf16.safetensors"
    hf_hub_download("Comfy-Org/Qwen-Image-2.1", f"text_encoders/{TE}", local_dir=M)

!ls -lh {M}/diffusion_models {M}/text_encoders {M}/vae | grep -v "put_"


In [ ]:
#@title 4. ComfyUI をバックグラウンドで起動
import subprocess, time, urllib.request

log = open("/content/comfyui.log", "w")
comfy = subprocess.Popen(
    ["python", "main.py", "--listen", "127.0.0.1", "--port", "8188"],
    cwd="/content/ComfyUI", stdout=log, stderr=subprocess.STDOUT)

for _ in range(120):
    try:
        urllib.request.urlopen("http://127.0.0.1:8188/system_stats", timeout=2)
        print("ComfyUI 起動完了")
        break
    except Exception:
        time.sleep(2)
else:
    print("起動に失敗しました。ログ ↓")
    print(open("/content/comfyui.log").read()[-3000:])


In [ ]:
#@title 5. 画像を生成（ここを書き換えて何度でも実行）
prompt = "A cozy Japanese city hall service counter at dusk, warm lighting, a wooden sign that says 市民課" #@param {type:"string"}
negative = "oversaturated, overexposed, gibberish text" #@param {type:"string"}
width = 1024 #@param {type:"slider", min:512, max:1536, step:32}
height = 1024 #@param {type:"slider", min:512, max:1536, step:32}
steps = 25 #@param {type:"integer"}
seed = -1 #@param {type:"integer"}
#@markdown **文字入りモード**：前半を cfg 1.0、`switch_step` 以降を `cfg_text` で描き直して文字をくっきりさせる
text_mode = True #@param {type:"boolean"}
switch_step = 15 #@param {type:"integer"}
cfg_text = 3.0 #@param {type:"number"}
#@markdown 生成した画像を Google ドライブの `MyDrive/qwen_image/` にも保存する
save_to_drive = False #@param {type:"boolean"}

import json, random, time, shutil, urllib.request, urllib.parse
from IPython.display import Image, display

def build_workflow(prompt, negative, width, height, steps, seed, text_mode, switch_step, cfg_text, dit, te, vae):
    wf = {
        "1": {"class_type": "UnetLoaderGGUF", "inputs": {"unet_name": dit}},
        "2": {"class_type": "CLIPLoader", "inputs": {"clip_name": te, "type": "qwen_image", "device": "default"}},
        "3": {"class_type": "VAELoader", "inputs": {"vae_name": vae}},
        "4": {"class_type": "TextEncodeQwenImage21", "inputs": {
            "clip": ["2", 0], "prompt": prompt, "negative_prompt": negative, "resolution": 1024}},
        "5": {"class_type": "EmptyLatentImage", "inputs": {"width": width, "height": height, "batch_size": 1}},
    }
    def ksampler(add_noise, cfg, start, end, leftover, latent):
        return {"class_type": "KSamplerAdvanced", "inputs": {
            "model": ["1", 0], "add_noise": add_noise, "noise_seed": seed, "steps": steps, "cfg": cfg,
            "sampler_name": "euler", "scheduler": "simple",
            "positive": ["4", 0], "negative": ["4", 1], "latent_image": latent,
            "start_at_step": start, "end_at_step": end, "return_with_leftover_noise": leftover}}
    if text_mode:
        # 前半 cfg 1.0 で構図を決め、後半 cfg 3.0 ＋ネガティブで文字を描き直す
        wf["6"] = ksampler("enable", 1.0, 0, switch_step, "enable", ["5", 0])
        wf["7"] = ksampler("disable", cfg_text, switch_step, 10000, "disable", ["6", 0])
        last = "7"
    else:
        wf["6"] = ksampler("enable", 1.0, 0, 10000, "disable", ["5", 0])
        last = "6"
    wf["8"] = {"class_type": "VAEDecode", "inputs": {"samples": [last, 0], "vae": ["3", 0]}}
    wf["9"] = {"class_type": "SaveImage", "inputs": {"images": ["8", 0], "filename_prefix": "qwen21"}}
    return wf

API = "http://127.0.0.1:8188"

def api(path, data=None):
    req = urllib.request.Request(API + path, data=json.dumps(data).encode() if data is not None else None,
                                 headers={"Content-Type": "application/json"})
    try:
        return json.loads(urllib.request.urlopen(req).read())
    except urllib.error.HTTPError as e:
        raise RuntimeError(e.read().decode()) from None

if seed < 0:
    seed = random.randint(0, 2**32 - 1)
wf = build_workflow(prompt, negative, width, height, steps, seed, text_mode, switch_step, cfg_text, DIT, TE, VAE)

t0 = time.time()
pid = api("/prompt", {"prompt": wf})["prompt_id"]
print(f"seed={seed}  生成中…（初回はモデル読み込みで時間がかかります）")
while True:
    h = api(f"/history/{pid}")
    if pid in h:
        break
    time.sleep(2)

status = h[pid]["status"]
if status.get("status_str") != "success":
    for kind, msg in status.get("messages", []):
        if kind == "execution_error":
            print(f"エラー（{msg['node_type']}）: {msg['exception_message']}")
    print("詳しくは最後のログ確認セルを実行してください")
else:
    print(f"完了 {time.time() - t0:.0f} 秒")
    for img in h[pid]["outputs"]["9"]["images"]:
        path = f"/content/ComfyUI/output/{img['subfolder']}/{img['filename']}"
        display(Image(filename=path, width=768))
        if save_to_drive:
            from google.colab import drive
            import os
            if not os.path.exists("/content/drive"):
                drive.mount("/content/drive")
            os.makedirs("/content/drive/MyDrive/qwen_image", exist_ok=True)
            shutil.copy(path, f"/content/drive/MyDrive/qwen_image/{seed}_{img['filename']}")
            print("ドライブに保存しました")


## （任意）ComfyUI の画面を開く

Colab 標準のポート転送で ComfyUI の画面を別ウィンドウに開きます。ngrok などの外部トンネルは使いません。
**無料枠では「Web UI 主体の操作」が禁止**されているため、有料ユニット（Google AI プラン特典など）がある場合だけ使ってください。


In [ ]:
#@title 6.（任意）ComfyUI の画面を開く
from google.colab import output
output.serve_kernel_port_as_window(8188)


In [ ]:
#@title うまくいかないときのログ確認
!tail -n 60 /content/comfyui.log
